# Per-land-cover indicator table, burned pixels only

**Purpose** - recompute the per-land-cover indicator table over the pixels the
regression heads were actually trained on.

Averaging the model outputs over every test pixel mixes two populations: more than
90% of test pixels have no documented fire, and the regression heads still emit
non-zero values there even though the masked multi-task loss gave them no gradient
on those pixels. Restricting the table to pixels with at least one documented fire
month in the fire-reference inventory reports the indicators over the population
they are defined on.

**Main inputs** - `predictions_test.npz` and `targets_aux.npz` from notebook 05 and
notebook 04, the land-cover one-hot features, and `config.FILES`.

**Main outputs** - `outputs/tables/table_4_5_full_burned.csv`,
`table_4_5_test_burned.csv`, `table_4_5_sample_sizes.csv` and
`table_4_5_revised_burned_only.md`. The `4_5` in these filenames is the table number
from the draft this notebook was written against; the number changed in later drafts
but the filenames were kept so the committed outputs stay traceable.

**Role in the thesis workflow** - produces the per-land-cover indicator table in the
Results chapter. It assigns each pixel its land-cover class from the current-year
one-hot features at month 0, whereas `metrics.per_lc_stats` uses `lc_dominant`, the
modal class over all 108 months. Section 17 of `notebooks/08_target_diagnostics.ipynb`
audits that difference and reports how much it changes the per-class counts.

**Reported per class** - the number of burned test pixels, the analytical target means,
and the corresponding model-output means.

In [1]:
# --- Setup ---
import sys, os
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    THESIS_ROOT = '/content/drive/MyDrive/thesisv4'
else:
    THESIS_ROOT = os.path.expanduser('~/thesisv4')

sys.path.insert(0, THESIS_ROOT)

import numpy as np
import pandas as pd
import config

print('THESIS_ROOT:', THESIS_ROOT)
print('Files available in models/main:')
for f in sorted(os.listdir(os.path.join(THESIS_ROOT, 'models/main'))):
    print(f'  {f}')

Mounted at /content/drive
THESIS_ROOT: /content/drive/MyDrive/thesisv4
Files available in models/main:
  model_main.pt
  predictions_test.npz
  results_main.json
  training_history.json


In [2]:
# --- Locate the required files ---
# The exact paths depend on config.FILES; adjust if needed.
print('Config file paths:')
for key, val in config.FILES.items():
    print(f'  {key:20s} = {val}')

Config file paths:
  burn_count_tif       = /content/drive/MyDrive/thesisv4/data/raw/burn_count_with_mask_10m.tif
  pixel_coords         = /content/drive/MyDrive/thesisv4/data/raw/pixel_coords_10m.csv
  raw_timeseries       = /content/drive/MyDrive/thesisv4/data/raw/full_pixels_10m.csv
  selected_features    = /content/drive/MyDrive/thesisv4/data/eda/selected_features.json
  X_scaled             = /content/drive/MyDrive/thesisv4/data/processed/X_scaled.npy
  y_burned             = /content/drive/MyDrive/thesisv4/data/processed/y_burned.npy
  targets_aux          = /content/drive/MyDrive/thesisv4/data/processed/targets_aux.npz
  splits               = /content/drive/MyDrive/thesisv4/data/processed/splits.json
  scaler               = /content/drive/MyDrive/thesisv4/data/processed/scaler.pkl
  feature_names        = /content/drive/MyDrive/thesisv4/data/processed/feature_names.json
  model_main           = /content/drive/MyDrive/thesisv4/models/main/model_main.pt
  history_main         = 

In [3]:
# --- Load model predictions on the test set ---
preds_test_path = config.FILES['preds_test']
if not os.path.isabs(preds_test_path):
    preds_test_path = os.path.join(THESIS_ROOT, preds_test_path)

print(f'Loading predictions from: {preds_test_path}')
preds = np.load(preds_test_path)

print('\nContents of predictions_test.npz:')
for k in preds.files:
    arr = preds[k]
    if arr.ndim == 0:
        print(f'  {k}: scalar = {arr.item()}')
    else:
        print(f'  {k}: shape={arr.shape}, dtype={arr.dtype}')

# Test pixel indices in the original array
idx_test = preds['indices']
print(f'\nTest pixel indices: {idx_test.shape[0]:,} pixels')

# Model outputs - averaged across time dimension
# (each pixel gets one severity/persistence/recovery_gap value)
# Check shapes to understand structure
print('\nModel output arrays:')
print(f'  severity:     {preds["severity"].shape}')
print(f'  persistence:  {preds["persistence"].shape}')
print(f'  recovery_gap: {preds["recovery_gap"].shape}')
print(f'  burned_probs: {preds["burned_probs"].shape}')

Loading predictions from: /content/drive/MyDrive/thesisv4/models/main/predictions_test.npz

Contents of predictions_test.npz:
  severity: shape=(91339,), dtype=float32
  persistence: shape=(91339,), dtype=float32
  recovery_gap: shape=(91339,), dtype=float32
  burned_probs: shape=(9864612,), dtype=float32
  burned_labels: shape=(9864612,), dtype=float32
  mask: shape=(9864612,), dtype=bool
  indices: shape=(91339,), dtype=int64
  threshold: scalar = 0.502462311557789

Test pixel indices: 91,339 pixels

Model output arrays:
  severity:     (91339,)
  persistence:  (91339,)
  recovery_gap: (91339,)
  burned_probs: (9864612,)


In [4]:
# --- Load analytical targets (derived indicators, one value per pixel) ---
# The targets file contains: severity, persistence, recovery_gap, recurrence, burned
# as ANALYTICAL values (not model outputs), for the full labeled sample.

targets_path = config.FILES.get('targets_aux')
if targets_path and not os.path.isabs(targets_path):
    targets_path = os.path.join(THESIS_ROOT, targets_path)

# Fall back to a common location if config doesn't have it
# targets_aux.npz is the correct file per config.FILES

print(f'Loading targets from: {targets_path}')
targets = np.load(targets_path)

print('\nContents of targets file:')
for k in targets.files:
    arr = targets[k]
    print(f'  {k}: shape={arr.shape}, dtype={arr.dtype}, '
          f'min={arr.min():.3f}, max={arr.max():.3f}')

Loading targets from: /content/drive/MyDrive/thesisv4/data/processed/targets_aux.npz

Contents of targets file:
  recurrence: shape=(608922,), dtype=float32, min=0.000, max=1.000
  severity: shape=(608922,), dtype=float32, min=0.000, max=1.000
  persistence: shape=(608922,), dtype=float32, min=0.000, max=1.000
  recovery_gap: shape=(608922,), dtype=float32, min=0.000, max=1.000


In [5]:
# --- Extract per-pixel LC labels from the feature array ---
# Config structure:
#   LC_CODES        = [10, 11, 20, 30, 40, 41, 50, 60, 61, 80]  (10 classes)
#   LC_FEATURES     = LC_FEATURES_PREV (10 cols) + LC_FEATURES_CURR (10 cols) = 20 cols
#   MODEL_FEATURES  = RAW_BANDS + INDICES + zscores + anomalies + LC_FEATURES + DELTA + HARMONIC
#
# The pixel's actual land-cover class is in LC_FEATURES_CURR (last 10 of the 20 LC cols).
# LC_FEATURES_PREV is the previous year LC (historical context feature).

# Compute LC_COLS from MODEL_FEATURES positions
LC_COLS_CURR = [config.MODEL_FEATURES.index(name) for name in config.LC_FEATURES_CURR]
print(f'LC_CODES:            {config.LC_CODES}')
print(f'LC_FEATURES_CURR:    {config.LC_FEATURES_CURR}')
print(f'LC_COLS_CURR (index in MODEL_FEATURES): {LC_COLS_CURR}')

# Load the feature array
features_path = config.FILES['X_scaled']
print(f'\nLoading feature array from: {features_path}')
X_all = np.load(features_path, mmap_mode='r')
print(f'Feature array shape: {X_all.shape}')

# LC values are constant across time; read at t=0
lc_one_hot = np.array(X_all[:, 0, LC_COLS_CURR])   # (N, 10) one-hot
print(f'lc_one_hot shape: {lc_one_hot.shape}')

# argmax gives the position (0..9) within LC_CODES
lc_idx = np.argmax(lc_one_hot, axis=1)

# Map position to LC code
lc_code_all = np.array([config.LC_CODES[i] for i in lc_idx])
print(f'lc_code_all shape: {lc_code_all.shape}')
print(f'\nPixel counts per LC code:')
for code in config.LC_CODES:
    n = int((lc_code_all == code).sum())
    name = config.LC_CATEGORIES[code]
    print(f'  {code:3d} {name:22s} n = {n:>8,}')

# Sanity: how many pixels have a valid (non-zero-sum) one-hot?
row_sums = lc_one_hot.sum(axis=1)
n_zero = int((row_sums == 0).sum())
if n_zero > 0:
    print(f'\nWarning: {n_zero:,} pixels have all-zero LC one-hot (may be scaled to zero).')
    print('If this number is large, LC extraction may be unreliable and needs adjustment.')


LC_CODES:            [10, 11, 20, 30, 40, 41, 50, 60, 61, 80]
LC_FEATURES_CURR:    ['lc_curr_forest', 'lc_curr_orchards', 'lc_curr_shrubland', 'lc_curr_grassland', 'lc_curr_cropland', 'lc_curr_covered_agriculture', 'lc_curr_built_up', 'lc_curr_bare', 'lc_curr_fallow', 'lc_curr_water']
LC_COLS_CURR (index in MODEL_FEATURES): [36, 37, 38, 39, 40, 41, 42, 43, 44, 45]

Loading feature array from: /content/drive/MyDrive/thesisv4/data/processed/X_scaled.npy
Feature array shape: (608922, 108, 52)
lc_one_hot shape: (608922, 10)
lc_code_all shape: (608922,)

Pixel counts per LC code:
   10 forest                 n =   48,984
   11 orchards               n =   22,734
   20 shrubland              n =   80,829
   30 grassland              n =  163,187
   40 cropland               n =  196,169
   41 covered_agriculture    n =    5,740
   50 built_up               n =    4,424
   60 bare                   n =   66,436
   61 fallow                 n =   18,161
   80 water                  n =    2,25

In [6]:
# --- LC code -> name mapping (already have lc_code_all with raw LC codes) ---
LC_CODE_TO_NAME = {code: config.LC_CATEGORIES[code].replace("_", " ").title()
                    for code in config.LC_CODES}
MAIN_CODES = config.LC_CODES   # [10, 11, 20, 30, 40, 41, 50, 60, 61, 80]

print(f'LC codes and names (display order matches config.LC_CODES):')
for code in MAIN_CODES:
    print(f'  {code:3d} -> {LC_CODE_TO_NAME[code]}')


LC codes and names (display order matches config.LC_CODES):
   10 -> Forest
   11 -> Orchards
   20 -> Shrubland
   30 -> Grassland
   40 -> Cropland
   41 -> Covered Agriculture
   50 -> Built Up
   60 -> Bare
   61 -> Fallow
   80 -> Water


In [7]:
# --- Identify burned pixels (from analytical target of recurrence or burned mask) ---
# A pixel is BURNED if it experienced at least one fire in the fire-reference inventory.
# In the targets file, this corresponds to:
#   - 'burned' array > 0, OR
#   - 'recurrence' > 0 (equivalent, since recurrence = min(fire_count/5, 1))

# Try 'burned' first, fall back to 'recurrence'
if 'burned' in targets.files:
    burned_all = targets['burned'] > 0
    print("Using targets['burned'] > 0 to identify burned pixels.")
elif 'recurrence' in targets.files:
    burned_all = targets['recurrence'] > 0
    print("Using targets['recurrence'] > 0 to identify burned pixels.")
else:
    raise KeyError('Neither burned nor recurrence in targets file.')

n_burned_all = int(burned_all.sum())
n_total = burned_all.shape[0]
print(f'\nBurned pixels in FULL labeled sample: {n_burned_all:,} / {n_total:,} '
      f'({100*n_burned_all/n_total:.2f}%)')

# Burned pixels within the TEST set only
burned_test = burned_all[idx_test]
n_burned_test = int(burned_test.sum())
print(f'Burned pixels in TEST subset:         {n_burned_test:,} / {idx_test.shape[0]:,} '
      f'({100*n_burned_test/idx_test.shape[0]:.2f}%)')

Using targets['recurrence'] > 0 to identify burned pixels.

Burned pixels in FULL labeled sample: 295,006 / 608,922 (48.45%)
Burned pixels in TEST subset:         44,303 / 91,339 (48.50%)


In [8]:
# --- Prepare per-pixel model output means ---
# Based on actual shapes:
#   severity, persistence, recovery_gap: (91339,) - already per-pixel
#   burned_probs, burned_labels, mask:   (9864612,) - per pixel-month, N*T=91339*108
# For burned_probs, aggregate to per-pixel by taking the max (ever-burned indicator).

N_test = 91339
T = 108   # months

mask_test = preds['mask'].reshape(N_test, T)
burned_probs_2d = preds['burned_probs'].reshape(N_test, T)

# Per-pixel max burned probability (matches the burned_prob raster's semantics)
brn_pred_pp = np.where(mask_test, burned_probs_2d, 0).max(axis=1)

# Regression outputs already per-pixel
sev_pred_pp = preds['severity']
per_pred_pp = preds['persistence']
rec_pred_pp = preds['recovery_gap']

print(f'Per-pixel model output arrays:')
print(f'  severity_pred:    shape={sev_pred_pp.shape}, mean={np.nanmean(sev_pred_pp):.3f}')
print(f'  persistence_pred: shape={per_pred_pp.shape}, mean={np.nanmean(per_pred_pp):.3f}')
print(f'  recovery_pred:    shape={rec_pred_pp.shape}, mean={np.nanmean(rec_pred_pp):.3f}')
print(f'  burned_prob_pred: shape={brn_pred_pp.shape}, mean={np.nanmean(brn_pred_pp):.3f}')


Per-pixel model output arrays:
  severity_pred:    shape=(91339,), mean=0.424
  persistence_pred: shape=(91339,), mean=0.550
  recovery_pred:    shape=(91339,), mean=0.415
  burned_prob_pred: shape=(91339,), mean=0.421


In [9]:
# --- Build a per-pixel DataFrame for the TEST set ---
df_test = pd.DataFrame({
    'lc_code':        lc_code_all[idx_test],
    'burned':         burned_test,
    'sev_target':     targets['severity'][idx_test],
    'per_target':     targets['persistence'][idx_test],
    'rec_target':     targets['recovery_gap'][idx_test],
    'sev_pred':       sev_pred_pp,
    'per_pred':       per_pred_pp,
    'rec_pred':       rec_pred_pp,
    'burned_prob':    brn_pred_pp,
})

print(f'Test-set DataFrame built: {len(df_test):,} rows')
print(f'Burned in test: {df_test["burned"].sum():,}')

# Filter to burned pixels only
df_test_burned = df_test[df_test['burned']].copy()
print(f'\nTest-set BURNED subset: {len(df_test_burned):,} rows')

Test-set DataFrame built: 91,339 rows
Burned in test: 44,303

Test-set BURNED subset: 44,303 rows


In [10]:
# --- Compute the per-LC-class table on TEST BURNED ---
def compute_lc_table(df, code_to_name, main_codes):
    rows = []
    for code in main_codes:
        sub = df[df['lc_code'] == code]
        if len(sub) == 0:
            rows.append({
                'LC code':        code,
                'LC name':        code_to_name.get(code, '?'),
                'n_burned':       0,
                'sev_target':     np.nan,
                'per_target':     np.nan,
                'rec_target':     np.nan,
                'sev_pred':       np.nan,
                'per_pred':       np.nan,
                'rec_pred':       np.nan,
                'burned_prob':    np.nan,
            })
            continue
        rows.append({
            'LC code':        code,
            'LC name':        code_to_name.get(code, '?'),
            'n_burned':       len(sub),
            'sev_target':     round(float(sub['sev_target'].mean()), 3),
            'per_target':     round(float(sub['per_target'].mean()), 3),
            'rec_target':     round(float(sub['rec_target'].mean()), 3),
            'sev_pred':       round(float(sub['sev_pred'].mean()),   3),
            'per_pred':       round(float(sub['per_pred'].mean()),   3),
            'rec_pred':       round(float(sub['rec_pred'].mean()),   3),
            'burned_prob':    round(float(sub['burned_prob'].mean()),3),
        })
    return pd.DataFrame(rows)

table_test = compute_lc_table(df_test_burned, LC_CODE_TO_NAME, MAIN_CODES)

print('=' * 100)
print('TABLE (test-set burned pixels only)')
print('=' * 100)
print(table_test.to_string(index=False))

small_classes_test = table_test[table_test['n_burned'] < 30]
print(f'\nTotal burned test pixels: {int(table_test["n_burned"].sum()):,}')
if len(small_classes_test) > 0:
    print(f'Classes with n < 30 (interpret with caution or merge/drop):')
    for _, row in small_classes_test.iterrows():
        print(f'  {int(row["LC code"]):3d} {row["LC name"]:22s} n = {int(row["n_burned"])}')

TABLE (test-set burned pixels only)
 LC code             LC name  n_burned  sev_target  per_target  rec_target  sev_pred  per_pred  rec_pred  burned_prob
      10              Forest      5561       0.592       0.594       0.252     0.597     0.602     0.260        0.688
      11            Orchards       554       0.210       0.301       0.152     0.204     0.305     0.157        0.444
      20           Shrubland      9652       0.635       0.627       0.500     0.644     0.630     0.508        0.635
      30           Grassland     18325       0.662       0.588       0.606     0.670     0.594     0.615        0.683
      40            Cropland      7308       0.345       0.570       0.472     0.352     0.577     0.485        0.513
      41 Covered Agriculture        45       0.487       0.494       0.207     0.431     0.515     0.214        0.393
      50            Built Up       114       0.456       0.529       0.374     0.501     0.549     0.366        0.540
      60            

In [11]:
# --- Alternative: compute on FULL labeled sample (all burned pixels) ---
# Model outputs are only available for the test set; for the full sample, we
# can only report the ANALYTICAL TARGETS (which are defined for all labeled
# pixels). Model predictions on training / validation pixels would be biased
# (they were used to fit the model), so we do not report them here.

df_full = pd.DataFrame({
    'lc_code':    lc_code_all,
    'burned':     burned_all,
    'sev_target': targets['severity'],
    'per_target': targets['persistence'],
    'rec_target': targets['recovery_gap'],
})

df_full_burned = df_full[df_full['burned']].copy()

def compute_lc_table_full(df, code_to_name, main_codes):
    rows = []
    for code in main_codes:
        sub = df[df['lc_code'] == code]
        if len(sub) == 0:
            rows.append({
                'LC code':    code,
                'LC name':    code_to_name.get(code, '?'),
                'n_burned':   0,
                'sev_target': np.nan,
                'per_target': np.nan,
                'rec_target': np.nan,
            })
            continue
        rows.append({
            'LC code':    code,
            'LC name':    code_to_name.get(code, '?'),
            'n_burned':   len(sub),
            'sev_target': round(float(sub['sev_target'].mean()), 3),
            'per_target': round(float(sub['per_target'].mean()), 3),
            'rec_target': round(float(sub['rec_target'].mean()), 3),
        })
    return pd.DataFrame(rows)

table_full = compute_lc_table_full(df_full_burned, LC_CODE_TO_NAME, MAIN_CODES)

print('=' * 100)
print('TABLE (full labeled sample, all burned pixels — analytical targets only)')
print('=' * 100)
print(table_full.to_string(index=False))

print(f'\nTotal burned pixels in full sample: {int(table_full["n_burned"].sum()):,}')

TABLE (full labeled sample, all burned pixels — analytical targets only)
 LC code             LC name  n_burned  sev_target  per_target  rec_target
      10              Forest     37914       0.593       0.598       0.255
      11            Orchards      3872       0.207       0.312       0.156
      20           Shrubland     64429       0.636       0.628       0.505
      30           Grassland    121880       0.662       0.586       0.606
      40            Cropland     48533       0.343       0.570       0.476
      41 Covered Agriculture       284       0.389       0.421       0.185
      50            Built Up       821       0.453       0.539       0.325
      60                Bare      5783       0.481       0.654       0.409
      61              Fallow     10010       0.607       0.547       0.569
      80               Water      1480       0.558       0.343       0.035

Total burned pixels in full sample: 295,006


In [12]:
# --- Side-by-side count comparison ---
comp = pd.DataFrame({
    'LC code':          MAIN_CODES,
    'LC name':          [LC_CODE_TO_NAME[c] for c in MAIN_CODES],
    'n_burned_test':    [int(table_test[table_test['LC code'] == c]['n_burned'].iloc[0]) for c in MAIN_CODES],
    'n_burned_full':    [int(table_full[table_full['LC code'] == c]['n_burned'].iloc[0]) for c in MAIN_CODES],
})
comp['ratio_test_full'] = (comp['n_burned_test'] / comp['n_burned_full']).round(3)

print('=' * 80)
print('Per-class sample size comparison: test vs. full labeled sample')
print('=' * 80)
print(comp.to_string(index=False))
print('\nUse the FULL table for LC classes where test n < 30.')
print('Use the TEST table for LC classes where test n >= 30.')

Per-class sample size comparison: test vs. full labeled sample
 LC code             LC name  n_burned_test  n_burned_full  ratio_test_full
      10              Forest           5561          37914            0.147
      11            Orchards            554           3872            0.143
      20           Shrubland           9652          64429            0.150
      30           Grassland          18325         121880            0.150
      40            Cropland           7308          48533            0.151
      41 Covered Agriculture             45            284            0.158
      50            Built Up            114            821            0.139
      60                Bare            895           5783            0.155
      61              Fallow           1608          10010            0.161
      80               Water            241           1480            0.163

Use the FULL table for LC classes where test n < 30.
Use the TEST table for LC classes where test n 

In [13]:
# --- Export markdown tables and CSVs for the thesis ---
OUT_DIR = os.path.join(THESIS_ROOT, 'outputs', 'tables')
os.makedirs(OUT_DIR, exist_ok=True)

# Markdown
md_test = table_test.to_markdown(index=False, floatfmt='.3f')
md_full = table_full.to_markdown(index=False, floatfmt='.3f')
md_comp = comp.to_markdown(index=False)

md_all = (
    '# Table 4.5 (revised) — Per-LC indicators, burned pixels only\n\n'
    '## (a) Test set, burned pixels only\n\n'
    'Both analytical targets (target) and model predictions (pred) are shown.\n\n'
    + md_test + '\n\n'
    '## (b) Full labeled sample, burned pixels only\n\n'
    'Only analytical targets are shown; model predictions on training / '
    'validation pixels would be biased.\n\n'
    + md_full + '\n\n'
    '## (c) Sample size comparison\n\n'
    + md_comp + '\n\n'
    '_Rounded to 3 decimals. Classes with n < 30 in the test set '
    'should be interpreted with caution or reported from the full sample._\n'
)

md_path = os.path.join(OUT_DIR, 'table_4_5_revised_burned_only.md')
with open(md_path, 'w') as f:
    f.write(md_all)
print(f'Markdown saved: {md_path}')

# CSVs
table_test.to_csv(os.path.join(OUT_DIR, 'table_4_5_test_burned.csv'), index=False)
table_full.to_csv(os.path.join(OUT_DIR, 'table_4_5_full_burned.csv'), index=False)
comp.to_csv(os.path.join(OUT_DIR, 'table_4_5_sample_sizes.csv'), index=False)
print('CSVs saved to:', OUT_DIR)

print('\n' + '=' * 80)
print('MARKDOWN OUTPUT (copy for thesis):')
print('=' * 80)
print(md_all)

Markdown saved: /content/drive/MyDrive/thesisv4/outputs/tables/table_4_5_revised_burned_only.md
CSVs saved to: /content/drive/MyDrive/thesisv4/outputs/tables

MARKDOWN OUTPUT (copy for thesis):
# Table 4.5 (revised) — Per-LC indicators, burned pixels only

## (a) Test set, burned pixels only

Both analytical targets (target) and model predictions (pred) are shown.

|   LC code | LC name             |   n_burned |   sev_target |   per_target |   rec_target |   sev_pred |   per_pred |   rec_pred |   burned_prob |
|----------:|:--------------------|-----------:|-------------:|-------------:|-------------:|-----------:|-----------:|-----------:|--------------:|
|        10 | Forest              |       5561 |        0.592 |        0.594 |        0.252 |      0.597 |      0.602 |      0.260 |         0.688 |
|        11 | Orchards            |        554 |        0.210 |        0.301 |        0.152 |      0.204 |      0.305 |      0.157 |         0.444 |
|        20 | Shrubland           | 